# GomokuAI: VS Code + Colab GPU smoke

Use the official Google Colab extension and select a Colab GPU kernel. This notebook performs only a tiny smoke run. Long training requires separate approval. Local Windows files are not automatically present in the remote runtime. Copy this development checkout to Drive first; preserve the original notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys
PROJECT_DIR = Path('/content/drive/MyDrive/GomokuAI')
assert (PROJECT_DIR / 'alphazero').is_dir(), 'Copy the development checkout to Drive first'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
%pip install -r requirements-export.txt


In [ ]:
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a Colab GPU kernel; no GPU run has occurred'
print(torch.cuda.get_device_name(0))


In [ ]:
import json
from alphazero.config import AlphaZeroConfig
from alphazero.pipeline import run_training_loop
settings = json.loads(Path('configs/colab-smoke.json').read_text())
settings['checkpoint_dir'] = '/content/drive/MyDrive/GomokuAI-runs/vscode-smoke'
config = AlphaZeroConfig(**settings)
print(config)
run_training_loop(config, resume=True)


## Resume
After iteration 0 finishes, setting total_iterations=2 and rerunning with resume=True adds one tiny iteration. Do not set a large iteration count without approval. latest.pt is the authoritative atomic snapshot.


In [ ]:
%run -m scripts.benchmark_inference --device cuda --output /content/drive/MyDrive/GomokuAI-runs/inference-gpu.json


## Export after smoke
Run scripts.export_model with the persistent latest.pt and a NEW output filename. It verifies ONNX Runtime parity. Connect the resulting .onnx and .json in the web training panel. A smoke model is not a strong player. See docs/TRAINING.md.
